In [23]:
import torch
from utils import load_splits
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset

In [24]:
#global random seed
SEED = 42
torch.manual_seed(SEED)

In [25]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

True
NVIDIA GeForce RTX 3050 6GB Laptop GPU


device(type='cuda')

In [26]:
X_train, X_val, X_test, y_train, y_val, y_test = load_splits()

1
2
3


In [27]:
MODEL_NAME = "distilbert-base-uncased"
def init_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label={0: "True", 1:"Fake"},
        label2id={"True": 0, "Fake": 0}
    )
    return model


In [28]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize(batch):

    _tokenizer = tokenizer(
        batch["text"],
        truncation = True,
        max_length = 256
    )
    return _tokenizer

def tokenize_dataset(x, y):
    dict = {
        "text" : x.astype(str).tolist(),
        "labels" : y.astype(int).tolist()
    }
    dataset = Dataset.from_dict(dict)
    tokenized_dataset = dataset.map(tokenize, batched=True)
    return tokenized_dataset


In [15]:
ds_train, ds_val = tokenize_dataset(X_train, y_train), tokenize_dataset(X_val, y_val)



Map: 100%|██████████| 9333/9333 [00:00<00:00, 30906.46 examples/s]


In [29]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
    }

In [ ]:
from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)

import gc

def training_loop(learning_rate):
    print(f"training with learning rate {learning_rate}")

    model = init_model()
    training_args = TrainingArguments(
        output_dir=f"models/distilbert_{learning_rate}",
        learning_rate=learning_rate,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=5,
        weight_decay=0.01,
        fp16=torch.cuda.is_available(),
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_total_limit=2,
        logging_steps=100,
        report_to="none",
        seed=42,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=ds_train,
        eval_dataset=ds_val,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
    )
    trainer.train()
    for log in trainer.state.log_history:
        if "eval_f1_macro" in log:
            results.append({
                "learning_rate": learning_rate,
                "epoch": log["epoch"],
                "val_loss": log["eval_loss"],
                "val_accuracy": log["eval_accuracy"],
                "val_precision": log["eval_precision_macro"],
                "val_recall": log["eval_recall_macro"],
                "val_f1": log["eval_f1_macro"],
            })

    del trainer, model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results
